# M6 — Prédiction de churn ou CLV

Modèles prédictifs sur la fidélité (churn) ou la valeur client (CLV), à partir de features RFM (Récence, Fréquence, Montant).

> ⚠️ **Limite du jeu de données** : seulement 5 clients dans l'échantillon fourni. Les modèles ci-dessous sont donc **illustratifs du pipeline** (feature engineering → entraînement → évaluation) et non des résultats statistiquement fiables. Avec un volume réel (centaines/milliers de clients), le même code produirait des scores exploitables.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, mean_absolute_error
from src.data_loader import load_sales_with_context

df = load_sales_with_context()
df.head()

,Sale_ID,Product_ID,Customer_ID,Date,Quantity,Sale_Price,Channel,Name,Age,Gender,Location,Join_Date,Total_Spent,Product_Name,Category,Price,Brand
0,1,101,2001,2023-01-15,2,50.0,Online,Alice,28,Female,New York,2022-05-10,500.0,T-shirt,Clothing,25.0,Brand A
1,2,102,2002,2023-01-16,1,75.0,In-Store,Bob,35,Male,Los Angeles,2022-06-15,750.0,Jeans,Clothing,75.0,Brand B
2,3,103,2001,2023-01-17,3,30.0,Online,Alice,28,Female,New York,2022-05-10,500.0,Sneakers,Footwear,30.0,Brand C
3,4,104,2003,2023-01-18,1,120.0,In-Store,Charlie,22,Male,Chicago,2022-07-20,300.0,Jacket,Outerwear,120.0,Brand D
4,5,105,2004,2023-01-19,2,45.0,Online,Diana,30,Female,Houston,2022-08-25,600.0,Hat,Accessories,22.5,Brand E


## Construction des features RFM (Récence, Fréquence, Montant)

- **Récence** : nombre de jours entre le dernier achat du client et la date la plus récente observée dans le jeu de données.
- **Fréquence** : nombre de transactions du client.
- **Montant** : dépense totale du client (`Total_Spent`).

In [2]:
snapshot_date = df["Date"].max()

rfm = df.groupby("Customer_ID").agg(
    Age=("Age", "first"),
    Last_Purchase=("Date", "max"),
    Frequency=("Sale_ID", "count"),
    Monetary=("Total_Spent", "first"),
).reset_index()
rfm["Recency"] = (snapshot_date - rfm["Last_Purchase"]).dt.days
rfm = rfm.drop(columns="Last_Purchase")
rfm

,Customer_ID,Age,Frequency,Monetary,Recency
0,2001,28,2,500.0,2
1,2002,35,1,750.0,3
2,2003,22,1,300.0,1
3,2004,30,1,600.0,0


## Prédiction de la CLV (régression)

Cible : `Monetary` (proxy de la CLV). Features : `Age`, `Frequency`, `Recency`.

In [3]:
X = rfm[["Age", "Frequency", "Recency"]]
y_clv = rfm["Monetary"]

clv_model = LinearRegression()
clv_model.fit(X, y_clv)
rfm["CLV_Predicted"] = clv_model.predict(X)
print("MAE (in-sample, échantillon trop petit pour un vrai train/test split) :",
      mean_absolute_error(y_clv, rfm["CLV_Predicted"]))
rfm[["Customer_ID", "Monetary", "CLV_Predicted"]]

MAE (in-sample, échantillon trop petit pour un vrai train/test split) : 1.1368683772161603e-13


,Customer_ID,Monetary,CLV_Predicted
0,2001,500.0,500.0
1,2002,750.0,750.0
2,2003,300.0,300.0
3,2004,600.0,600.0


## Prédiction du risque de churn (classification)

Label illustratif : client "à risque" si sa récence est supérieure à la médiane de l'échantillon (proxy simple, à remplacer par une règle métier réelle — ex. pas d'achat depuis 90 jours).

In [4]:
rfm["At_Risk"] = (rfm["Recency"] > rfm["Recency"].median()).astype(int)

X_churn = rfm[["Age", "Frequency", "Monetary"]]
y_churn = rfm["At_Risk"]

churn_model = RandomForestClassifier(random_state=42, n_estimators=100)
churn_model.fit(X_churn, y_churn)
rfm["At_Risk_Predicted"] = churn_model.predict(X_churn)
print("Accuracy (in-sample) :", accuracy_score(y_churn, rfm["At_Risk_Predicted"]))
rfm[["Customer_ID", "Recency", "At_Risk", "At_Risk_Predicted"]]

Accuracy (in-sample) : 1.0


,Customer_ID,Recency,At_Risk,At_Risk_Predicted
0,2001,2,1,1
1,2002,3,1,1
2,2003,1,0,0
3,2004,0,0,0


## Importance des variables (churn)

In [5]:
pd.Series(churn_model.feature_importances_, index=X_churn.columns).sort_values(ascending=False)

Age          0.498693
Monetary     0.266667
Frequency    0.234641
dtype: float64

## Conclusion

Le pipeline RFM → CLV / churn est fonctionnel de bout en bout. **À refaire avec un volume de données réel** (train/test split, validation croisée, comparaison Random Forest vs XGBoost vs régression logistique) pour obtenir des scores fiables exploitables en M7.